# Domaća zadaća 1: Metaheuristika i optimizacija hiperparametara SVM modela

**Univerzitet u Zenici**  
**Politehnički fakultet**  
**Odsjek:** Softversko inženjerstvo  
**Predmet:** Rudarenje podataka  
**Vježba br. 5**

**Student:** Delaida Muminović  
**Broj indeksa:** 138

---

Cilj zadaće je proširiti prostor hiperparametara SVM modela koristeći razumno odabrane vrijednosti, završiti Random Search koji je započet na posljednjim vježbama, te implementirati Bat algoritam (algoritam šišmiša) za pretragu hiperparametara. Na kraju se porede rezultati i vrijeme izvršavanja obje metode. Koristi se Iris skup podataka.

## Uvoz potrebnih biblioteka

Koriste se standardne Python biblioteke za obradu podataka (`pandas`, `numpy`),
SVM model iz `scikit-learn`, te `matplotlib` za vizualizacije i poređenja.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## Učitavanje skupa podataka

Učitava se Iris CSV datoteka koja sadrži podatke o cvjetovima. Kolone uključuju `sepal_length`, `sepal_width`, `petal_length`, `petal_width` i vrstu cvijeta (`species`). Zadatak je klasifikacija vrste cvijeta na osnovu fizičkih karakteristika.

In [ ]:
df = pd.read_csv("iris.csv")

X = df.drop("species", axis=1)
y = df["species"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Dimenzije skupa podataka: {X.shape}")
print(f"Broj klasa: {y.nunique()}")
print(f"Klase: {y.unique().tolist()}")
display(X.head())

## Definisanje proširenog, ali kontrolisanog prostora hiperparametara

Umjesto ekstremno velikih raspona koji usporavaju izvršavanje, koristi se smislen i dovoljno širok prostor pretrage. Time i dalje istražujemo različite vrijednosti hiperparametara, ali zadržavamo brzo i stabilno izvršavanje notebooka.

In [ ]:
OPCIJE = {
    "svc__C": np.logspace(-2, 2, 20).tolist(),
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": np.logspace(-3, 1, 20).tolist(),
    "svc__tol": [1e-3, 1e-4],
}

total = 1
for key, values in OPCIJE.items():
    total *= len(values)
    print(f"  {key.replace('svc__', '')}: {len(values)} vrijednosti")

print(f"\nUkupan broj kombinacija: {total:,}")
print("Random Search je praktičan jer pretražuje reprezentativan uzorak ovog prostora.")

## Random Search sa optimizovanim opcijama

Random Search nasumično bira kombinacije hiperparametara umjesto da isprobava sve. Da bi izvršavanje bilo brzo i stabilno, koristi se `Pipeline` sa `StandardScaler`, uži skup kernela (`linear`, `rbf`) i `n_iter=30`. To daje dobar balans između brzine i kvaliteta rezultata.

In [ ]:
start_time = time.time()

svm_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("svc", SVC(max_iter=1000)),
    ]
)

random_search = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=OPCIJE,
    n_iter=30,
    scoring="f1_weighted",
    cv=cv,
    random_state=42,
    n_jobs=1
)
random_search.fit(X, y)

random_time = time.time() - start_time

print("Najbolji parametri (Random Search):")
for param, value in random_search.best_params_.items():
    print(f"  {param.replace('svc__', '')}: {value}")
print(f"\nNajbolji F1 score: {random_search.best_score_:.4f}")
print(f"Vrijeme izvršavanja: {random_time:.2f} sekundi")
print(f"Broj evaluacija: {random_search.n_iter}")

## Bat algoritam (algoritam šišmiša) - Pseudokod

Pseudokod sa vježbi:

```
za svaki bat i:
    beta = random(0, 1)
    f[i] = f_min + (f_max - f_min) * beta          # frekvencija
    v[i] = v[i] + (x[i] - best) * f[i]             # brzina
    x_new = x[i] + v[i]                            # nova pozicija

    ako random(0,1) > pulse_rate[i]:
        x_new = best + epsilon * average_loudness  # lokalna pretraga

    x_new = popravi_granice(x_new)                 # ograničenja
    fitness_new = objective(x_new)                 # evaluacija

    ako fitness_new < fitness[i] i random(0,1) < loudness[i]:
        x[i] = x_new
        fitness[i] = fitness_new
        loudness[i] = alpha * loudness[i]          # smanji glasnoću
```

Svaki "šišmiš" predstavlja jednu kombinaciju hiperparametara. Pozicija šišmiša u prostoru je zapravo skup hiperparametara za SVM model.

## Bat algoritam - Implementacija pomoćnih funkcija

Svaki šišmiš ima poziciju u kontinuiranom prostoru sa 3 dimenzije:
- **dim 0** - log10(C), opseg od 0.01 do 100
- **dim 1** - log10(gamma), opseg od 0.001 do 10
- **dim 2** - indeks kernela (0-1, mapiranje na `linear` i `rbf`)

Logaritamska skala omogućava da pretraga ostane široka, ali numerički stabilna. Funkcija `objective` evaluira SVM model pomoću 5-fold cross-validation procedure, a rezultati se keširaju kako bi se izbjeglo ponavljanje istih evaluacija.

In [ ]:
# Granice pretrage za svaku dimenziju: [log10(C), log10(gamma), kernel_index]
LOWER_BOUNDS = np.array([-2.0, -3.0, 0.0])
UPPER_BOUNDS = np.array([2.0, 1.0, 1.0])

KERNEL_MAP = ["linear", "rbf"]
OBJECTIVE_CACHE = {}


def dekodiranje_pozicije(pozicija):
    """Pretvara kontinuiranu poziciju šišmiša u hiperparametre SVM modela."""
    log_c = float(np.clip(pozicija[0], LOWER_BOUNDS[0], UPPER_BOUNDS[0]))
    log_gamma = float(np.clip(pozicija[1], LOWER_BOUNDS[1], UPPER_BOUNDS[1]))
    kernel_idx = int(np.clip(round(pozicija[2]), 0, len(KERNEL_MAP) - 1))
    kernel = KERNEL_MAP[kernel_idx]

    return {
        "C": 10 ** log_c,
        "gamma": 10 ** log_gamma if kernel == "rbf" else "scale",
        "kernel": kernel,
    }


def popravi_granice(pozicija):
    """Vraća poziciju unutar dozvoljenih granica."""
    return np.clip(pozicija, LOWER_BOUNDS, UPPER_BOUNDS)


def objective(pozicija, X, y):
    """Evaluira SVM model s datim hiperparametrima.
    Vraća negativan F1 score jer Bat algoritam minimizira, a mi želimo maksimizirati F1."""
    params = dekodiranje_pozicije(pozicija)
    gamma_key = params["gamma"] if isinstance(params["gamma"], str) else round(params["gamma"], 6)
    key = (round(params["C"], 6), gamma_key, params["kernel"])

    if key not in OBJECTIVE_CACHE:
        model = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "svc",
                    SVC(
                        C=params["C"],
                        gamma=params["gamma"],
                        kernel=params["kernel"],
                        tol=1e-3,
                        max_iter=1000,
                    ),
                ),
            ]
        )
        score = cross_val_score(model, X, y, cv=cv, scoring="f1_weighted", n_jobs=1).mean()
        OBJECTIVE_CACHE[key] = -score

    return OBJECTIVE_CACHE[key]


print("Pomoćne funkcije su definisane.")

## Bat algoritam - Glavna funkcija

Implementacija Bat algoritma prati pseudokod sa vježbi, ali koristi manju populaciju i manje iteracija kako bi notebook ostao brz. Time se zadržava ideja metaheurističke pretrage, a ukupno vrijeme izvršavanja ostaje razumno.

In [ ]:
def bat_algorithm(X, y, n_bats=8, max_iter=15, f_min=0, f_max=2, alpha=0.9):
    """
    Bat algoritam za optimizaciju hiperparametara SVM modela.

    Parametri:
        n_bats    - broj šišmiša (veličina populacije)
        max_iter  - broj iteracija
        f_min     - minimalna frekvencija
        f_max     - maksimalna frekvencija
        alpha     - faktor smanjenja glasnoće
    """
    n_dim = len(LOWER_BOUNDS)

    # Inicijalizacija populacije - nasumične pozicije unutar granica
    x = np.random.uniform(LOWER_BOUNDS, UPPER_BOUNDS, size=(n_bats, n_dim))
    v = np.zeros((n_bats, n_dim))          # brzine pocetno 0

    loudness = np.ones(n_bats) * 0.9        # početna glasnoća
    pulse_rate = np.ones(n_bats) * 0.1      # početna stopa pulsa

    # Evaluacija pocetne populacije
    fitness = np.array([objective(x[i], X, y) for i in range(n_bats)])

    # Pronalaženje najboljeg rješenja
    best_idx = np.argmin(fitness)
    best = x[best_idx].copy()
    best_fitness = fitness[best_idx]

    # Za praćenje napretka
    history = [(-best_fitness)]
    eval_count = n_bats

    print(f"Početno najbolje rješenje: F1 = {-best_fitness:.4f}")

    # Glavna petlja
    for t in range(max_iter):
        avg_loudness = np.mean(loudness)

        for i in range(n_bats):
            # Ažuriranje frekvencije
            beta = np.random.random()
            frequency = f_min + (f_max - f_min) * beta

            # Ažuriranje brzine
            v[i] = v[i] + (x[i] - best) * frequency

            # Nova kandidatska pozicija
            x_new = x[i] + v[i]

            # Lokalna pretraga - ako random > pulse_rate, skoči blizu najboljeg
            if np.random.random() > pulse_rate[i]:
                epsilon = np.random.uniform(-1, 1, n_dim)
                x_new = best + epsilon * avg_loudness

            # Popravi granice
            x_new = popravi_granice(x_new)

            # Evaluacija nove pozicije
            fitness_new = objective(x_new, X, y)
            eval_count += 1

            # Prihvatanje novog rješenja
            if fitness_new < fitness[i] and np.random.random() < loudness[i]:
                x[i] = x_new
                fitness[i] = fitness_new
                loudness[i] = alpha * loudness[i]

            # Ažuriraj globalno najbolje rješenje
            if fitness[i] < best_fitness:
                best = x[i].copy()
                best_fitness = fitness[i]

        history.append(-best_fitness)

        if (t + 1) % 5 == 0:
            print(f"  Iteracija {t+1}/{max_iter}: najbolji F1 = {-best_fitness:.4f}")

    best_params = dekodiranje_pozicije(best)
    return best_params, -best_fitness, history, eval_count


print("Bat algoritam je definisan.")

## Pokretanje Bat algoritma

Pokreće se Bat algoritam sa 8 šišmiša i 15 iteracija. Ukupan broj evaluacija SVM modela je 8 + 8 * 15 = 128, što je znatno brže od prethodne verzije.

In [ ]:
np.random.seed(42)
OBJECTIVE_CACHE.clear()

start_time = time.time()
bat_params, bat_score, bat_history, bat_evals = bat_algorithm(X, y, n_bats=8, max_iter=15)
bat_time = time.time() - start_time

print(f"\nNajbolji parametri (Bat algoritam):")
for param, value in bat_params.items():
    print(f"  {param}: {value}")
print(f"\nNajbolji F1 score: {bat_score:.4f}")
print(f"Vrijeme izvršavanja: {bat_time:.2f} sekundi")
print(f"Ukupan broj evaluacija: {bat_evals}")

## Poređenje rezultata: Random Search vs Bat algoritam

Poredimo obje metode po tri kriterija: kvalitet rješenja (F1 score), vrijeme izvršavanja i broj evaluacija modela.

In [ ]:
rezultati = pd.DataFrame({
    "Metoda": ["Random Search", "Bat algoritam"],
    "F1 Score": [random_search.best_score_, bat_score],
    "Vrijeme (s)": [round(random_time, 2), round(bat_time, 2)],
    "Broj evaluacija": [random_search.n_iter, bat_evals]
})

print("=" * 60)
print("POREĐENJE METODA")
print("=" * 60)
display(rezultati)
print("=" * 60)

score_diff = abs(bat_score - random_search.best_score_)
print(f"\nRazlika u F1 score-u: {score_diff:.4f}")

## Vizualizacija konvergencije Bat algoritma

Graf prikazuje kako se najbolji F1 score Bat algoritma mijenja kroz iteracije. Crvena isprekidana linija predstavlja rezultat Random Searcha za poređenje.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(len(bat_history)), bat_history, marker='o', markersize=3, color='steelblue', label='Bat algoritam')
plt.axhline(y=random_search.best_score_, color='red', linestyle='--', label=f'Random Search ({random_search.best_score_:.4f})')
plt.xlabel('Iteracija')
plt.ylabel('Najbolji F1 Score')
plt.title('Konvergencija Bat algoritma vs Random Search')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Vizualizacija poređenja metoda

Direktno poređenje F1 score-a i vremena izvršavanja obje metode putem bar grafova.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metode = ['Random Search', 'Bat algoritam']
scores = [random_search.best_score_, bat_score]
vremena = [random_time, bat_time]

# F1 score
axes[0].bar(metode, scores, color=['#e74c3c', '#2ecc71'])
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Poređenje F1 score-a')
axes[0].set_ylim(min(scores) - 0.05, 1.01)
for i, v in enumerate(scores):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

# Vrijeme
axes[1].bar(metode, vremena, color=['#e74c3c', '#2ecc71'])
axes[1].set_ylabel('Vrijeme (sekundi)')
axes[1].set_title('Poređenje vremena izvršavanja')
for i, v in enumerate(vremena):
    axes[1].text(i, v + 0.1, f'{v:.2f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Zaključak

- **Random Search** nasumično bira kombinacije iz kontrolisanog prostora hiperparametara i daje stabilne rezultate uz kratko vrijeme izvršavanja.
- **Bat algoritam** inteligentno pretražuje prostor koristeći mehanizme frekvencije, brzine i glasnoće, pa kroz iteracije konvergira ka dobrim rješenjima.
- Zahvaljujući standardizaciji podataka, užem prostoru pretrage i keširanju evaluacija, notebook se izvršava znatno brže nego ranije.
- Za sličan F1 score, obje metode mogu dati kvalitetna rješenja, ali Bat algoritam dodatno pokazuje ponašanje metaheurističke optimizacije kroz konvergenciju.